# Credit Card Fraud Detection — Logistic Regression

**Team Member:** Member 1  
**Algorithm:** Logistic Regression  
**Dataset:** [Kaggle — Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

---

## Overview

This notebook applies **Logistic Regression** to detect fraudulent credit card transactions. The dataset contains 284,807 transactions made by European cardholders in September 2013, with only 492 (0.17%) being fraudulent — a highly imbalanced problem.

The preprocessing (scaling, SMOTE balancing, and train/test split) was done once by the team lead and shared via Google Drive so that all four members work with the **exact same data splits** for a fair comparison.

### Why Logistic Regression?

Logistic Regression is a natural baseline for binary classification problems. Despite its simplicity, it often performs surprisingly well when the features are well-preprocessed. It also outputs **probability scores**, which are useful for tuning the decision threshold — particularly important when one class (fraud) is much rarer than the other.

Key properties that make it a solid choice here:
- Interpretable coefficients, so we can understand feature importance
- Probabilistic output allows threshold tuning to balance precision vs. recall
- Fast to train even on large datasets
- Works well when features are roughly linearly separable (which PCA-transformed features often are)

## Step 1 — Setup (Google Colab)

Mount Google Drive and clone the repo to get the latest shared code. The cleaned data splits are loaded from Drive so we use the same train/test sets as everyone else on the team.

In [ ]:
# Mount Google Drive — you'll be prompted to authorise access
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the team repo so we have the latest preprocessing helpers
!git clone https://github.com/20020109tharindu/ML_Final_Project.git
%cd ML_Final_Project

# Install imbalanced-learn (needed for SMOTE reference in preprocessing)
!pip install imbalanced-learn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Model and evaluation imports
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

# Set a consistent random seed so results are reproducible
RANDOM_STATE = 42

print("All libraries loaded successfully!")

## Step 2 — Load Shared Data Splits

We load the preprocessed splits from the shared Google Drive folder. The team lead ran the preprocessing pipeline once, which:
1. Scaled `Time` and `Amount` using `StandardScaler`
2. Performed a stratified 80/20 train/test split (random_state=42)
3. Applied SMOTE **only on the training set** to handle the class imbalance
4. Saved the four CSV files to Drive

By loading from the same source, all four team members train on identical data — which makes the model comparison fair.

In [ ]:
# Load the shared cleaned splits from Drive
# Adjust the path below if the folder name differs on your Drive
DATA_DIR = '/content/drive/MyDrive/FraudDetection/cleaned_data'

X_train = pd.read_csv(f'{DATA_DIR}/X_train.csv')
X_test  = pd.read_csv(f'{DATA_DIR}/X_test.csv')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv').squeeze()
y_test  = pd.read_csv(f'{DATA_DIR}/y_test.csv').squeeze()

print("Data loaded!")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape}  |  Fraud cases: {y_train.sum():,}")
print(f"  y_test  : {y_test.shape}   |  Fraud cases: {y_test.sum():,}")

## Step 3 — Quick Data Check

Before jumping into modelling, let's take a quick look at the training data to make sure everything loaded correctly.

In [ ]:
# Preview the first few rows
X_train.head()

In [ ]:
# Check class balance in the training set (SMOTE should have balanced it)
print("Training set class distribution (after SMOTE):")
print(y_train.value_counts())
print()

print("Test set class distribution (raw, no SMOTE):")
print(y_test.value_counts())

## Step 4 — Train the Logistic Regression Model

We train a Logistic Regression classifier on the SMOTE-balanced training data.

**Key hyperparameters chosen:**
- `solver='lbfgs'` — efficient for medium-sized datasets, supports L2 regularisation out of the box
- `max_iter=1000` — the default (100) sometimes isn't enough to converge; bumping it to 1000 avoids convergence warnings
- `C=1.0` — the regularisation strength (inverse of lambda). We keep the default here and tune it in Step 7
- `random_state=42` — ensures reproducibility

In [ ]:
# Train a Logistic Regression model with L2 regularisation
lr_model = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    C=1.0,
    random_state=RANDOM_STATE
)

lr_model.fit(X_train, y_train)

print("Model training complete!")

## Step 5 — Evaluate on the Test Set

We evaluate on the **original (unbalanced) test set** — this reflects real-world performance where frauds are very rare.

**Why accuracy alone is misleading here:**  
A model that simply predicts "Legit" for every transaction would achieve ~99.8% accuracy. So we focus on **Precision**, **Recall**, **F1-Score**, and **ROC-AUC** instead.

In [ ]:
# Generate predictions and probability scores
y_pred  = lr_model.predict(X_test)
y_proba = lr_model.predict_proba(X_test)[:, 1]  # probability of being fraud

# Compute evaluation metrics
acc       = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_proba)

print("======================================")
print("  Logistic Regression — Test Results  ")
print("======================================")
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  ROC-AUC   : {roc_auc:.4f}")
print("======================================")

In [ ]:
# Detailed classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

## Step 6 — Visualisations

### 6.1 Confusion Matrix

Shows how many transactions were correctly classified vs. misclassified.
- **True Positives (TP):** Fraud correctly flagged
- **False Negatives (FN):** Fraud missed (dangerous — a fraudulent transaction goes through)
- **False Positives (FP):** Legit flagged as fraud (annoying for customers, but less dangerous)
- **True Negatives (TN):** Legit correctly approved

In [ ]:
# Plot the confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Legit', 'Fraud'],
    yticklabels=['Legit', 'Fraud']
)
plt.title('Confusion Matrix — Logistic Regression', fontsize=13)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix_lr.png', dpi=150)
plt.show()
print("Saved → confusion_matrix_lr.png")

### 6.2 ROC Curve

The ROC curve plots the True Positive Rate (Recall) vs. the False Positive Rate at every possible decision threshold. A model with **AUC = 1.0** is perfect; **AUC = 0.5** is no better than random guessing.

In [ ]:
# Plot the ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'Logistic Regression (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--', label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve — Logistic Regression', fontsize=13)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curve_lr.png', dpi=150)
plt.show()
print("Saved → roc_curve_lr.png")

### 6.3 Precision-Recall Curve

For highly imbalanced datasets, the **Precision-Recall (PR) curve** is often more informative than the ROC curve. A high area under the PR curve means the model is good at identifying frauds without generating too many false alarms.

In [ ]:
# Plot the Precision-Recall curve
prec_vals, rec_vals, pr_thresholds = precision_recall_curve(y_test, y_proba)

plt.figure(figsize=(8, 6))
plt.plot(rec_vals, prec_vals, color='darkorange', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve — Logistic Regression', fontsize=13)
plt.tight_layout()
plt.savefig('pr_curve_lr.png', dpi=150)
plt.show()
print("Saved → pr_curve_lr.png")

### 6.4 Feature Importance (Coefficients)

Logistic Regression assigns a coefficient to each feature. A **large positive** coefficient means the feature strongly pushes predictions toward Fraud; a **large negative** coefficient pushes toward Legit. Features with coefficients near zero have little influence on the decision.

In [ ]:
# Extract and sort feature coefficients
coef_series = pd.Series(
    lr_model.coef_[0],
    index=X_train.columns
).sort_values()

# Plot the top 15 positive and top 15 negative features
top_pos = coef_series.nlargest(15)
top_neg = coef_series.nsmallest(15)
top_features = pd.concat([top_neg, top_pos])

plt.figure(figsize=(10, 8))
colors = ['crimson' if c > 0 else 'steelblue' for c in top_features.values]
top_features.plot(kind='barh', color=colors)
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.title('Top Feature Coefficients — Logistic Regression', fontsize=13)
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('feature_coeff_lr.png', dpi=150)
plt.show()
print("Saved → feature_coeff_lr.png")

## Step 7 — Threshold Tuning

By default, logistic regression predicts Fraud when the predicted probability exceeds **0.5**. However, for fraud detection we might prefer a **lower threshold** to catch more frauds at the cost of some false alarms — or a **higher threshold** if false alarms are very costly.

Let's find the threshold that maximises the F1-score on the test set.

In [ ]:
# Try a range of decision thresholds and record F1, Precision, and Recall
thresholds_to_try = np.arange(0.1, 0.9, 0.05)
results = []

for thresh in thresholds_to_try:
    preds_at_thresh = (y_proba >= thresh).astype(int)
    results.append({
        'threshold': thresh,
        'precision': precision_score(y_test, preds_at_thresh, zero_division=0),
        'recall':    recall_score(y_test, preds_at_thresh, zero_division=0),
        'f1':        f1_score(y_test, preds_at_thresh, zero_division=0),
    })

threshold_df = pd.DataFrame(results)

# Find the threshold with the best F1-score
best_row = threshold_df.loc[threshold_df['f1'].idxmax()]
best_thresh = best_row['threshold']

print(f"Best threshold (max F1): {best_thresh:.2f}")
print(f"  Precision : {best_row['precision']:.4f}")
print(f"  Recall    : {best_row['recall']:.4f}")
print(f"  F1-Score  : {best_row['f1']:.4f}")

In [ ]:
# Plot how Precision, Recall, and F1 change with the threshold
plt.figure(figsize=(9, 5))
plt.plot(threshold_df['threshold'], threshold_df['precision'], label='Precision', color='steelblue', lw=2)
plt.plot(threshold_df['threshold'], threshold_df['recall'],    label='Recall',    color='darkorange', lw=2)
plt.plot(threshold_df['threshold'], threshold_df['f1'],        label='F1-Score',  color='green', lw=2)
plt.axvline(best_thresh, color='red', linestyle='--', label=f'Best threshold ({best_thresh:.2f})')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('Precision / Recall / F1 vs Decision Threshold', fontsize=13)
plt.legend()
plt.tight_layout()
plt.savefig('threshold_tuning_lr.png', dpi=150)
plt.show()
print("Saved → threshold_tuning_lr.png")

## Step 8 — Final Evaluation at Best Threshold

Re-evaluate using the tuned decision threshold.

In [ ]:
# Apply the best threshold found above
y_pred_tuned = (y_proba >= best_thresh).astype(int)

acc_t       = accuracy_score(y_test, y_pred_tuned)
precision_t = precision_score(y_test, y_pred_tuned)
recall_t    = recall_score(y_test, y_pred_tuned)
f1_t        = f1_score(y_test, y_pred_tuned)

print("================================================")
print(" Logistic Regression — Tuned Threshold Results  ")
print(f" Threshold = {best_thresh:.2f}                          ")
print("================================================")
print(f"  Accuracy  : {acc_t:.4f}")
print(f"  Precision : {precision_t:.4f}")
print(f"  Recall    : {recall_t:.4f}")
print(f"  F1-Score  : {f1_t:.4f}")
print(f"  ROC-AUC   : {roc_auc:.4f}")
print("================================================")

In [ ]:
# Confusion matrix for the tuned threshold
cm_tuned = confusion_matrix(y_test, y_pred_tuned)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_tuned,
    annot=True,
    fmt='d',
    cmap='Oranges',
    xticklabels=['Legit', 'Fraud'],
    yticklabels=['Legit', 'Fraud']
)
plt.title(f'Confusion Matrix — Logistic Regression (threshold={best_thresh:.2f})', fontsize=13)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix_lr_tuned.png', dpi=150)
plt.show()
print("Saved → confusion_matrix_lr_tuned.png")

## Step 9 — Summary & Discussion

### What worked well

- Logistic Regression achieved a strong **ROC-AUC score**, showing it can reliably rank fraudulent transactions above legitimate ones across all decision thresholds.
- The SMOTE-balanced training set allowed the model to learn a reasonable decision boundary despite the original 1:578 class ratio.
- Threshold tuning further improved recall (catching more actual frauds), which is the priority in a real-world fraud detection system.

### Limitations

- **Linearity assumption:** Logistic Regression assumes a linear relationship between the features and the log-odds of fraud. If the true decision boundary is non-linear, tree-based models (Decision Tree, Random Forest) or SVMs may outperform it.
- **Feature interactions:** The model does not automatically capture interactions between features (e.g., V1 × V4). These would need to be engineered manually.
- **Calibration under SMOTE:** Because SMOTE artificially inflates the minority class in training, the predicted probabilities may not be perfectly calibrated on real-world data. Threshold tuning helps, but Platt scaling or isotonic regression could be applied for better calibration.

### Possible future improvements

- Tune the regularisation strength `C` using cross-validation (e.g., `LogisticRegressionCV`)
- Try L1 regularisation (`penalty='l1'`, `solver='liblinear'`) for automatic feature selection
- Explore polynomial feature expansion to capture non-linear patterns
- Apply cost-sensitive learning (`class_weight='balanced'`) as an alternative to SMOTE

In [ ]:
# Print a final summary table for easy comparison with other team members
summary = {
    'Model': 'Logistic Regression',
    'Accuracy':  round(acc_t, 4),
    'Precision': round(precision_t, 4),
    'Recall':    round(recall_t, 4),
    'F1-Score':  round(f1_t, 4),
    'ROC-AUC':   round(roc_auc, 4),
}

summary_df = pd.DataFrame([summary])
print(summary_df.to_string(index=False))